# GAIA Translator — RAG Evals

This notebook is a hands-on tour of how we measure the quality of our RAG pipeline and how *persona-aware* reranking changes what the LLM sees.

## What we measure and why

A RAG system can fail in three different places, and each needs its own metric:

| Failure mode | Symptom | Metric that catches it |
| --- | --- | --- |
| Retrieval misses the right paper | LLM has nothing to cite, hallucinates | **Recall@k**, **MRR** |
| Retrieval is dominated by one paper | LLM gives a one-sided translation | **Paper diversity** |
| Personalization signal isn't reaching retrieval | Persona shows in UI but doesn't change context | **Persona drift**, **concept coverage** |
| LLM cites things the retrieval didn't support | Inline `[n]` references fabricated | **Citation faithfulness** |

**Why these and not embedding similarity alone?** Cosine distance is a *confidence* signal, not a *correctness* signal. A query can produce 8 chunks with average cosine 0.85 and still miss the right paper — they're all wrong, just confidently wrong. Recall@k anchors quality to a hand-curated ground truth.

## How to use this notebook

1. From `backend/`, run `python -m app.eval.run --out eval_results.json` (or `--with-llm` for citation evals).
2. Set `RESULTS_PATH` below.
3. Run all cells.

The notebook is intentionally annotated like a teaching artifact — read the markdown cells, don't just stare at the charts.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

RESULTS_PATH = Path('../backend/eval_results.json')
assert RESULTS_PATH.exists(), f'Run `python -m app.eval.run --out {RESULTS_PATH}` first.'
raw = json.loads(RESULTS_PATH.read_text())
results = raw['results']
print(f'Loaded {len(results)} eval items.')

## 1. Headline retrieval metrics

Three numbers per query, baseline retrieval (no persona).

- **Recall@8**: 1 if any gold paper appears in the top 8, else 0. *The* most important metric.
- **MRR**: 1 / rank of the *first* gold chunk. Captures ordering — a gold paper at position 1 scores 1.0; at position 8 scores 0.125.
- **Paper diversity**: unique papers / 8. Low values are a red flag for translation tasks.

In [ ]:
rows = []
for r in results:
    rows.append({
        'item_id': r['item_id'],
        'query': r['query'][:60] + '…',
        'recall@8': r['baseline_metrics']['recall_at_k'],
        'precision@8': r['baseline_metrics']['precision_at_k'],
        'mrr': r['baseline_metrics']['mrr'],
        'paper_diversity': r['baseline_metrics']['paper_diversity'],
        'mean_cosine': r['baseline_metrics']['mean_cosine'],
    })
df = pd.DataFrame(rows)
df

In [ ]:
fig = make_subplots(rows=2, cols=2, subplot_titles=('Recall@8', 'MRR', 'Paper diversity', 'Mean cosine similarity'))
for i, (col, row, c) in enumerate([
    ('recall@8', 1, 1),
    ('mrr', 1, 2),
    ('paper_diversity', 2, 1),
    ('mean_cosine', 2, 2),
]):
    fig.add_trace(go.Bar(x=df['item_id'], y=df[col], name=col, showlegend=False), row=row, col=c)
fig.update_layout(height=600, title_text='Retrieval metrics per query — baseline (no persona)')
fig.show()

### How to read this chart

- A **bar at 0 on Recall@8** is a *failure*: the gold paper isn't in the prompt at all, so the LLM can only fabricate.
- High **mean cosine** with low **recall** = the retriever is confident but wrong (often: query is too generic or the gold paper uses very different vocabulary).
- Low **paper diversity** with high recall = we're finding the right paper but starving the LLM of cross-paper context. The diversity cap (`MAX_CHUNKS_PER_PAPER` in `retriever.py`) is the lever to fix this.

## 2. Persona impact: does personalization actually change retrieval?

Items that have a `persona` block are run twice — once without, once with. We compare:

- **Drift** = `1 − Jaccard(top-8 baseline, top-8 persona)`. **Drift = 0** means the persona did nothing to retrieval — a strong signal that either the persona fields are empty in tokens, or `PERSONA_WEIGHT` is too low.
- **Concept coverage** = fraction of persona-declared concept tokens that appear in at least one retrieved chunk. Higher = retrieval found content matching what the user said they care about.
- **Score gain** = average `(final_score − base_score)` over the persona run. Tells you the absolute magnitude of the rerank — orthogonal to whether it changed the *order*.

In [ ]:
persona_rows = []
for r in results:
    pm = r.get('persona_metrics')
    if pm:
        persona_rows.append({
            'item_id': r['item_id'],
            'drift': pm['drift_vs_baseline'],
            'concept_coverage': pm['concept_coverage'],
            'score_gain': pm['score_gain'],
            'recall@8 (persona)': pm['retrieval']['recall_at_k'],
            'recall@8 (baseline)': r['baseline_metrics']['recall_at_k'],
        })
pdf = pd.DataFrame(persona_rows)
pdf

In [ ]:
if not pdf.empty:
    fig = make_subplots(rows=1, cols=3, subplot_titles=('Persona drift (0=no change, 1=fully different)', 'Concept coverage', 'Avg score gain from rerank'))
    fig.add_trace(go.Bar(x=pdf['item_id'], y=pdf['drift'], marker_color='#4f46e5', showlegend=False), row=1, col=1)
    fig.add_trace(go.Bar(x=pdf['item_id'], y=pdf['concept_coverage'], marker_color='#16a34a', showlegend=False), row=1, col=2)
    fig.add_trace(go.Bar(x=pdf['item_id'], y=pdf['score_gain'], marker_color='#ea580c', showlegend=False), row=1, col=3)
    fig.update_layout(height=380, title_text='Personalization signal — is persona actually reaching retrieval?')
    fig.show()
else:
    print('No persona items in this run.')

### Interpreting these three together

| Drift | Concept coverage | What it means |
| --- | --- | --- |
| 0.0 | low | Persona text contains no tokens that match any chunk. Either user wrote vague things or vocabulary mismatch. |
| 0.0 | high | Top-8 is already concept-aligned without help. Persona is *redundant*, not broken. |
| >0.3 | high | Rerank is doing real work and surfacing the right content. **This is what we want.** |
| >0.3 | low | Rerank is reshuffling but on superficial tokens (e.g. matching 'python' everywhere). Worth tuning `PERSONA_WEIGHT` down or refining the persona schema. |

Score gain is a sanity check on magnitude — a non-zero drift with near-zero score gain means a tiebreaker-level reordering, not a substantive change.

## 3. Side-by-side: what changed in the top-8?

For each persona item, show which chunks dropped, which were added, and which stayed (just reordered). This is the most *concrete* way to see personalization at work.

In [ ]:
from IPython.display import HTML, display

def _key(c):
    return f"{c['paper_id']}::{c['text_excerpt'][:60]}"

for r in results:
    if not r.get('persona_metrics'):
        continue
    baseline = r['baseline_chunks']
    persona = r['chunks']  # this is the persona-rerank result
    base_keys = [_key(c) for c in baseline]
    pers_keys = [_key(c) for c in persona]
    dropped = [c for c in baseline if _key(c) not in pers_keys]
    added = [c for c in persona if _key(c) not in base_keys]
    kept_reordered = [c for c in persona if _key(c) in base_keys]
    
    html = [f"<h4>{r['item_id']} — <em>{r['query'][:120]}</em></h4>"]
    html.append("<table style='font-size:11px;width:100%;border-collapse:collapse'>")
    html.append("<tr><th style='text-align:left'>kept (reordered)</th><th style='text-align:left'>added by persona</th><th style='text-align:left'>dropped by persona</th></tr>")
    rows = max(len(kept_reordered), len(added), len(dropped))
    for i in range(rows):
        def fmt(lst, i, color):
            if i >= len(lst):
                return '<td></td>'
            c = lst[i]
            return f"<td style='border-top:1px solid #eee;padding:4px;color:{color}'><b>{(c['title'] or '?')[:50]}</b><br><small>{(c['text_excerpt'] or '')[:130]}…</small></td>"
        html.append('<tr>')
        html.append(fmt(kept_reordered, i, '#475569'))
        html.append(fmt(added, i, '#16a34a'))
        html.append(fmt(dropped, i, '#dc2626'))
        html.append('</tr>')
    html.append('</table>')
    display(HTML(''.join(html)))

## 4. Distance distribution

Histogram of cosine distances across all retrieved chunks. Useful for *calibrating* `PERSONA_WEIGHT`: if your distances cluster tightly (say 0.15–0.25), then `PERSONA_WEIGHT=0.35` will dominate every rerank — you probably want lower. If they're spread widely (0.1–0.6), small weights produce barely-visible reranks.

In [ ]:
all_dists = []
for r in results:
    for c in r['baseline_chunks']:
        all_dists.append(c['distance'])
fig = px.histogram(all_dists, nbins=30, title='Cosine distance distribution across all retrieved chunks')
fig.update_layout(showlegend=False, xaxis_title='cosine distance (lower = more similar)', yaxis_title='count')
fig.show()
print(f'min={min(all_dists):.3f}  median={sorted(all_dists)[len(all_dists)//2]:.3f}  max={max(all_dists):.3f}')

## 5. Citation faithfulness (only when run with `--with-llm`)

We split the rendered translation into sentences, find every `[n]` citation, and check that *some* chunk attributed to paper-n shares at least one **trigram** with the surrounding sentence. This is a deliberately *strict* signal: paraphrasing can drop the score even when the LLM was faithful, so 60-80% is a healthy range, not 100%.

If you want a stricter LLM-judge variant, replace `citation_faithfulness` in `metrics.py` with an LLM call ("does this claim follow from this passage?").

In [ ]:
cite_rows = []
for r in results:
    ce = r.get('citation_eval')
    if not ce:
        continue
    cite_rows.append({
        'item_id': r['item_id'],
        'n_cited': ce['n_cited'],
        'n_supported': ce['n_supported'],
        'fraction': ce['fraction'],
    })
if cite_rows:
    cdf = pd.DataFrame(cite_rows)
    fig = px.bar(cdf, x='item_id', y='fraction', title='Citation faithfulness — fraction of [n] citations grounded in retrieved chunks',
                 hover_data=['n_cited', 'n_supported'])
    fig.update_yaxes(range=[0, 1])
    fig.show()
    display(cdf)
else:
    print('No citation evals — re-run with `--with-llm` to populate this section.')

## 6. What to do with low numbers

A debugging playbook based on which metric is low.

**Low Recall@8 (the most common failure)**
1. Look at the actual top-8 chunks for the failing item (the dataframe in section 1 gives item_ids; cross-reference `results[i]['baseline_chunks']`).
2. If the gold paper has chunks at all in the database but none surfaced, vocabulary mismatch — try rewriting the query manually and rerun retrieval. If that fixes it, the fix is *query expansion* (an open task — see Phase B item #4 in the architecture doc).
3. If the gold paper has *no* chunks, ingestion is broken. Check `papers` and `chunks` tables.

**Low paper diversity**
- Lower `MAX_CHUNKS_PER_PAPER` in `app.rag.retriever`. Default 3; 2 is more aggressive.

**Zero persona drift**
- Print the persona dict — does it have actual text? Then print `_persona_tokens(persona)` from `retriever.py` — if the token set is small, words are getting filtered as stopwords.
- Raise `PERSONA_WEIGHT` (default 0.35). 0.5 makes rerank dominant; 0.1 makes it a tiebreaker.

**Low citation faithfulness with high recall**
- The LLM is paraphrasing well but losing token overlap — the trigram check is unfairly strict. Either accept it, or upgrade to an LLM-judge.
- Or the LLM is making claims that the retrieved chunks don't support — try lowering `temperature` (currently 0.3) and emphasizing 'only state facts present in retrieved literature' in the system prompt.